### Links and Resources
| Concept / Command | Description |
|-------------------|-------------|
| [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) | Canonical reference for streaming semantics |
| [Unsupported operations](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#unsupported-operations) | What you can't do on a streaming DataFrame |
| [Built-in functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) | `pyspark.sql.functions` — all stateless transforms |
| [`from_json`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html) | Parse a JSON string column into a struct |
| [`foreachBatch`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#using-foreach-and-foreachbatch) | Escape hatch — run any DataFrame op per micro-batch |

In [1]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# Streaming Transformations

Once you have a streaming DataFrame, what can you actually do with it? The short answer: **almost everything you do in batch — minus a handful of operations that don't make sense on an unbounded table.**

This notebook covers:

1. The transformations that work identically on streams and batches (the **stateless** ones).
2. The operations that are **forbidden or restricted** on streams, and why.
3. **`foreachBatch`** — the escape hatch that lets you run *any* batch operation per micro-batch.

Stateful transformations (windows, deduplication, joins) get their own notebooks (04 and 05).

## The mental model — same DataFrame, different lifecycle

A streaming DataFrame is just a `DataFrame` with `isStreaming = True`. The operators you call on it return new streaming DataFrames. Nothing executes until you call `.writeStream.start()`.

```python
df = spark.readStream.format("rate").load()    # streaming
df2 = df.filter("value > 10").withColumn("doubled", col("value") * 2)   # still streaming
df2.isStreaming                                # True
```

**Stateless** transformations operate on each row independently. They need no memory of past rows, scale trivially, and behave identically in batch and streaming. This is the bulk of what you'll write.

## A reusable demo source

We'll build a rate stream and shape it into something resembling event data — an `event_id`, a `user_id`, a `timestamp`, and a JSON `payload`. Every later cell transforms this stream and prints the output via the memory sink.

In [7]:
import time, uuid
from pyspark.sql import functions as F

# rate-micro-batch (not plain rate) so availableNow can terminate the stream — see notebook 02.
events = (spark.readStream
    .format("rate-micro-batch")
    .option("rowsPerBatch", 20)
    .option("numPartitions", 1)
    .load()
    .withColumnRenamed("value", "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int"))
    .withColumn("payload", F.to_json(F.struct(
        F.col("event_id").alias("id"),
        (F.rand() * 100).alias("score"),
        F.when(F.col("event_id") % 3 == 0, "premium").otherwise("free").alias("tier"),
    ))))

events.printSchema()
print("isStreaming:", events.isStreaming)

root
 |-- timestamp: timestamp (nullable = true)
 |-- event_id: long (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- payload: string (nullable = true)

isStreaming: True


### A small helper to peek at a streaming DataFrame

Throughout this notebook we'll write a stream into the **memory sink**, let `availableNow` drain it, and SELECT. Wrap it once.

In [6]:
def peek(stream_df, name, limit=10):
    """Run a streaming DF into the memory sink, await termination, return rows.
    Serverless rejects continuous triggers and the implicit temp checkpoint dir, so we use
    availableNow + a uuid-suffixed checkpointLocation. See notebook 02 for both constraints."""
    checkpoint = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/{name}_{uuid.uuid4().hex[:8]}"
    q = (stream_df.writeStream
         .format("memory")
         .queryName(name)
         .outputMode("append")
         .option("checkpointLocation", checkpoint)
         .trigger(availableNow=True)
         .start())
    q.awaitTermination()
    return spark.sql(f"SELECT * FROM {name} LIMIT {limit}")

## 1. Projection — `select`, `withColumn`, `drop`, `cast`

The bread-and-butter operations. They run row-by-row, no state, no surprises.

In [8]:
projected = (events
    .select("event_id", "user_id", "timestamp", "payload")
    .withColumn("event_id_str", F.col("event_id").cast("string"))
    .withColumn("hour", F.hour("timestamp"))
    .drop("payload"))

display(peek(projected, "projected_demo"))

+--------+-------+-------------------+------------+----+
|event_id|user_id|timestamp          |event_id_str|hour|
+--------+-------+-------------------+------------+----+
|0       |0      |1970-01-01 00:00:00|0           |0   |
|1       |1      |1970-01-01 00:00:00|1           |0   |
|2       |2      |1970-01-01 00:00:00|2           |0   |
|3       |3      |1970-01-01 00:00:00|3           |0   |
|4       |4      |1970-01-01 00:00:00|4           |0   |
|5       |0      |1970-01-01 00:00:00|5           |0   |
|6       |1      |1970-01-01 00:00:00|6           |0   |
|7       |2      |1970-01-01 00:00:00|7           |0   |
|8       |3      |1970-01-01 00:00:00|8           |0   |
|9       |4      |1970-01-01 00:00:00|9           |0   |
+--------+-------+-------------------+------------+----+



## 2. Filtering — `filter` / `where`

Identical to batch. Filter pushes through Catalyst the same way.

In [9]:
filtered = events.filter("user_id IN (0, 1)").select("event_id", "user_id")

display(peek(filtered, "filtered_demo"))

+--------+-------+
|event_id|user_id|
+--------+-------+
|0       |0      |
|1       |1      |
|5       |0      |
|6       |1      |
|10      |0      |
|11      |1      |
|15      |0      |
|16      |1      |
+--------+-------+



## 3. Parsing JSON payloads — `from_json`

Real streams almost always carry JSON or Avro payloads. Parse them with `from_json` and a schema, then unpack into columns. This is by far the most common pattern after Auto Loader / Kafka ingestion.

In [10]:
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType

payload_schema = StructType([
    StructField("id",    LongType()),
    StructField("score", DoubleType()),
    StructField("tier",  StringType()),
])

parsed = (events
    .withColumn("p", F.from_json("payload", payload_schema))
    .select("event_id", "user_id", "timestamp", "p.score", "p.tier"))

display(peek(parsed, "parsed_demo"))

+--------+-------+-------------------+------------------+-------+
|event_id|user_id|timestamp          |score             |tier   |
+--------+-------+-------------------+------------------+-------+
|0       |0      |1970-01-01 00:00:00|55.01346516887131 |premium|
|1       |1      |1970-01-01 00:00:00|66.23001567293359 |free   |
|2       |2      |1970-01-01 00:00:00|1.1107521735503778|free   |
|3       |3      |1970-01-01 00:00:00|17.728965326660195|premium|
|4       |4      |1970-01-01 00:00:00|41.13269866951865 |free   |
|5       |0      |1970-01-01 00:00:00|45.747048159960045|free   |
|6       |1      |1970-01-01 00:00:00|0.4361530678261616|premium|
|7       |2      |1970-01-01 00:00:00|95.51752704819317 |free   |
|8       |3      |1970-01-01 00:00:00|97.90107257336975 |free   |
|9       |4      |1970-01-01 00:00:00|14.791669219024438|premium|
+--------+-------+-------------------+------------------+-------+



## 4. Conditional logic — `when` / `otherwise`, `coalesce`, `case`

All standard. Use them to bucket, default, or normalize.

In [11]:
bucketed = (parsed
    .withColumn("score_bucket",
        F.when(F.col("score") < 33, "low")
         .when(F.col("score") < 66, "mid")
         .otherwise("high"))
    .withColumn("tier", F.coalesce("tier", F.lit("unknown"))))

display(peek(bucketed, "bucketed_demo"))

+--------+-------+-------------------+------------------+-------+------------+
|event_id|user_id|timestamp          |score             |tier   |score_bucket|
+--------+-------+-------------------+------------------+-------+------------+
|0       |0      |1970-01-01 00:00:00|77.51750648846222 |premium|high        |
|1       |1      |1970-01-01 00:00:00|32.20042714263843 |free   |low         |
|2       |2      |1970-01-01 00:00:00|12.101676604098976|free   |low         |
|3       |3      |1970-01-01 00:00:00|80.37715188819628 |premium|high        |
|4       |4      |1970-01-01 00:00:00|90.37160409745614 |free   |high        |
|5       |0      |1970-01-01 00:00:00|74.58549646625269 |free   |high        |
|6       |1      |1970-01-01 00:00:00|29.655468071898593|premium|low         |
|7       |2      |1970-01-01 00:00:00|25.771194892863658|free   |low         |
|8       |3      |1970-01-01 00:00:00|3.908849360121669 |free   |low         |
|9       |4      |1970-01-01 00:00:00|47.45753098801

## 5. Exploding arrays — `explode`

If a field is an array (think: list of items in an order, list of tags on an event) `explode` turns each element into its own row. Stateless and stream-safe.

In [12]:
with_tags = (events
    .withColumn("tags", F.array(
        F.lit("new"),
        F.when(F.col("event_id") % 2 == 0, F.lit("even")).otherwise(F.lit("odd")),
    )))

exploded = with_tags.select("event_id", F.explode("tags").alias("tag"))
display(peek(exploded, "exploded_demo"))

+--------+----+
|event_id|tag |
+--------+----+
|0       |new |
|0       |even|
|1       |new |
|1       |odd |
|2       |new |
|2       |even|
|3       |new |
|3       |odd |
|4       |new |
|4       |even|
+--------+----+



## 6. UDFs and Pandas UDFs

Plain Python UDFs work, but they cost: each row crosses the JVM↔Python boundary. Use a **Pandas UDF** (vectorised) when you have a non-trivial transformation. Both kinds work on streams.

In [17]:
# Catalyst built-in equivalent of a Pandas UDF — runs server-side without shipping Python,
# so it's safe under Databricks Connect regardless of client/server Python version.
#
# The pandas_udf form below is the textbook teaching example. It only runs when the local
# venv's Python *minor* version matches the serverless environment's; otherwise the server
# raises [PYTHON_VERSION_MISMATCH]. See CLAUDE.md for the constraint and how to probe it.
#
#   from pyspark.sql.functions import pandas_udf
#   import pandas as pd
#
#   @pandas_udf("string")
#   def label_user(uid: pd.Series) -> pd.Series:
#       return uid.map({0: "alice", 1: "bob", 2: "carol", 3: "dave", 4: "eve"}).fillna("unknown")
#
#   labeled = events.withColumn("user_name", label_user("user_id"))

# Catalyst-only version — no Python shipped to the cluster.
user_name_map = F.create_map(
    F.lit(0), F.lit("alice"),
    F.lit(1), F.lit("bob"),
    F.lit(2), F.lit("carol"),
    F.lit(3), F.lit("dave"),
    F.lit(4), F.lit("eve"),
)

labeled = (events
    .withColumn("user_name", F.coalesce(user_name_map[F.col("user_id")], F.lit("unknown")))
    .select("event_id", "user_id", "user_name"))
display(peek(labeled, "labeled_demo"))

+--------+-------+---------+
|event_id|user_id|user_name|
+--------+-------+---------+
|0       |0      |alice    |
|1       |1      |bob      |
|2       |2      |carol    |
|3       |3      |dave     |
|4       |4      |eve      |
|5       |0      |alice    |
|6       |1      |bob      |
|7       |2      |carol    |
|8       |3      |dave     |
|9       |4      |eve      |
+--------+-------+---------+



## What you cannot do on a streaming DataFrame

Some operations have no sensible meaning on an unbounded table. Spark rejects them at query analysis time.

| Operation | Why it's rejected |
|---|---|
| `df.count()` (action) | Would need to read the whole stream to a single number — never finishes |
| `df.show()`, `df.collect()`, `df.take(n)` | Same — these are batch actions; use the memory/console sink instead |
| `df.limit(n)` | Ambiguous: limit *across the whole stream*? Per batch? Spark refuses to guess |
| `df.distinct()` (without watermark) | Requires unbounded state |
| `df.sort()` / `df.orderBy()` | A global sort over an unbounded table is impossible |
| Multiple aggregations chained | `df.groupBy(a).count().groupBy(b).count()` — only one streaming aggregation per query |
| Most subqueries | Catalyst can't plan them streaming-safely |
| `df.foreach()` (the action) | Same reason as `collect` — but the **sink form** `writeStream.foreach()` is fine |

Rule of thumb: anything that requires Spark to know *the whole dataset* is illegal on a stream. Anything that processes *one row, one batch, or a windowed slice* is fine.

In [14]:
# Demonstration: this raises AnalysisException because limit isn't supported on streams.
# Use availableNow + an explicit checkpointLocation so we hit the limit analysis error,
# not the serverless trigger / temp-checkpoint errors first.
from pyspark.errors import AnalysisException

try:
    bad = events.limit(10)
    checkpoint = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/bad_demo_{uuid.uuid4().hex[:8]}"
    (bad.writeStream
        .format("memory")
        .queryName("bad_demo")
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .start())
except AnalysisException as e:
    print("rejected as expected:", str(e).split("\n")[0])
except Exception as e:
    print("rejected as expected:", type(e).__name__, str(e)[:200])

## Operations that work *with a watermark*

A few operations are unsafe in general but become valid once you bound how late data can arrive (a **watermark**, covered in notebook 04):

| Operation | With watermark |
|---|---|
| `dropDuplicates(["id"])` | Works — state for ids older than the watermark is evicted |
| `groupBy(window(...), key).agg(...)` | Works — windows older than the watermark are finalised and dropped |
| Stream-stream joins | Work — the watermark bounds how long unmatched rows are kept |

We'll use all three in the next two notebooks.

## `foreachBatch` — the escape hatch

Sometimes you genuinely need a batch operation inside a stream: `MERGE INTO` a Delta table, write the same data to two sinks, call an external API, run code that the streaming engine doesn't natively understand.

`foreachBatch` is the answer. The function you pass receives:

- `batch_df` — a **regular** (non-streaming) DataFrame containing this micro-batch's rows.
- `batch_id` — a monotonically increasing long, unique per query.

Inside the function, anything goes — `count()`, `collect()`, `MERGE`, multi-sink writes, etc.

In [18]:
# foreachBatch ships the Python function `show_batch` to the cluster, so it requires the
# local venv's Python *minor* version to match the serverless environment's. If they don't,
# the cluster raises [PYTHON_VERSION_MISMATCH] — wrapped here so the notebook stays usable.
# See CLAUDE.md "Pandas UDFs and foreachBatch need matching client/server Python".
from pyspark.errors.exceptions.connect import SparkConnectGrpcException

def show_batch(batch_df, batch_id):
    n = batch_df.count()                       # legal! batch_df is NOT streaming
    print(f"  batch {batch_id}: {n} rows")
    if n:
        batch_df.select("event_id", "user_id").show(3, truncate=False)

checkpoint = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/foreach_demo_{uuid.uuid4().hex[:8]}"
try:
    q = (events
         .select("event_id", "user_id")
         .writeStream
         .foreachBatch(show_batch)
         .outputMode("append")
         .option("checkpointLocation", checkpoint)
         .trigger(availableNow=True)
         .start())

    q.awaitTermination()
    print("done")
except SparkConnectGrpcException as e:
    msg = str(e)
    if "PYTHON_VERSION_MISMATCH" in msg or "different" in msg.lower():
        print("foreachBatch skipped — client/server Python minor versions differ.")
        print("Probe the server version with: spark.sql('SELECT current_version()').show(truncate=False)")
        print("Then install the matching Python minor version locally and recreate the venv.")
    else:
        raise

foreachBatch skipped — client/server Python minor versions differ.
Probe the server version with: spark.sql('SELECT current_version()').show(truncate=False)
Then install the matching Python minor version locally and recreate the venv.


**Caveats with `foreachBatch`:**

- The same `batch_id` may be re-delivered after a failure. To get **exactly-once**, your function must be **idempotent** (use `MERGE` keyed on `batch_id` or business key, not blind `INSERT`).
- You're now responsible for not blocking the executor for too long — long synchronous calls inside the function delay the next trigger.
- You lose the engine's per-sink optimisations. Don't reach for `foreachBatch` when `.toTable(...)` would work.

Notebook 08 has the production patterns: idempotent MERGE, multi-sink fan-out, error handling.

## A note on column references and DataFrame caching

Two batch habits that don't translate cleanly to streams:

- **`df.cache()` / `df.persist()`** — caching a streaming DataFrame is a no-op (or worse, an error in some Spark versions). The data is unbounded; there is nothing finite to cache. Cache the *output table* if you need fast reads.
- **`spark.sql("SELECT ... FROM streaming_view")`** — perfectly fine, as long as `streaming_view` was registered with `df.createOrReplaceTempView("streaming_view")` from a streaming DataFrame. The view inherits the streaming nature.

## Takeaways

- The **stateless** half of the DataFrame API — projection, filtering, JSON parsing, conditionals, explode, UDFs — works identically on streams. If the batch version is correct, the streaming version is correct.
- Operations that need to see the **whole dataset** (`limit`, `orderBy`, global `distinct`, multiple aggregations) are rejected at analysis time. This is a feature — it stops you writing queries that can never finish.
- Operations that need to **bound state** (deduplication, windowed aggregation, stream-stream joins) are legal *with a watermark* — see notebook 04.
- **`foreachBatch`** is the escape hatch when you need batch semantics inside a stream. Powerful, but you own idempotence.
- Don't `cache()` streams. There is nothing finite to hold.